# Sampling distributions and the central limit theorem
Sampling distributions built by simulation from five non-normal populations, the mean and variance the CLT predicts, the effect of the sample size, and a population where the CLT fails.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. A sampling distribution from a uniform population
1000 samples of size 30 from uniform(0, 1). Each row of `samples` is one sample.

In [ ]:
rng = np.random.default_rng(42)          # fixed seed
samples = rng.uniform(0, 1, size=(1000, 30))
print(samples.shape)                       # 1000 samples, 30 values each
# the mean of every row: the sampling distribution of the sample mean
sample_means = samples.mean(axis=1)
print(sample_means.shape)
print("mean of the means", round(sample_means.mean(), 4), " std of the means", round(sample_means.std(), 4))
print("CLT prediction: mean 0.5, std", round(np.sqrt(1 / 12 / 30), 4))

In [ ]:
fig = go.Figure(go.Histogram(x=sample_means, nbinsx=30))
fig.update_layout(template="simple_white", xaxis_title="sample mean", title="1000 means of samples of size 30")
fig.show()

## 2. A larger sample size gives a narrower, more normal shape

In [ ]:
means_300 = rng.uniform(0, 1, size=(1000, 300)).mean(axis=1)
print("n = 300: std of the means", round(means_300.std(), 4), " CLT:", round(np.sqrt(1 / 12 / 300), 4))

## 3. Four more populations: exponential, Poisson, gamma, binomial
Samples of size 50, 1000 samples each. The population is skewed; the skewness of the means is much closer to 0.

In [ ]:
populations = {
    "exponential(1)": stats.expon(),
    "Poisson(3)": stats.poisson(3),
    "gamma(2, 1)": stats.gamma(2),
    "binomial(10, 0.2)": stats.binom(10, 0.2),
}
for name, dist in populations.items():
    means = dist.rvs(size=(1000, 50), random_state=1).mean(axis=1)
    print(f"{name:18s} population skew {float(dist.stats(moments='s')):.2f}   means: average {means.mean():.3f}, "
          f"std {means.std():.4f} (CLT {dist.std() / np.sqrt(50):.4f}), skew {stats.skew(means):.2f}")

## 4. Checking the mean and the variance: gamma(2, 1), n = 50, 10,000 samples

In [ ]:
gamma = stats.gamma(2)                    # shape 2, scale 1
print("theoretical mean", gamma.mean(), " theoretical variance", gamma.var())
rng = np.random.default_rng(7)
means = rng.gamma(2, 1, size=(10_000, 50)).mean(axis=1)
print("mean of the sample means", round(means.mean(), 4))
print("variance of the sample means", round(means.var(), 4), " sigma^2 / n =", gamma.var() / 50)
print("standard error: empirical", round(means.std(), 4), " sigma / sqrt(n) =", round(gamma.std() / np.sqrt(50), 4))

## 5. The sample size, n = 1, 2, 5, 30, for an exponential population

In [ ]:
for n in [1, 2, 5, 30]:
    m = np.random.default_rng(3).exponential(1, (10_000, n)).mean(axis=1)
    print(f"n = {n:2d}: std of means {m.std():.3f}, sigma/sqrt(n) {1 / np.sqrt(n):.3f}, skewness {stats.skew(m):.2f}")

## 6. When the CLT fails: a Pareto population with alpha = 1.5 (infinite variance)

In [ ]:
for n in [30, 300, 3000]:
    m = (np.random.default_rng(5).pareto(1.5, (5000, n)) + 1).mean(axis=1)
    p1, p50, p99 = np.percentile(m, [1, 50, 99])
    print(f"n = {n:4d}: 1st, 50th, 99th percentile of the means {p1:.2f}, {p50:.2f}, {p99:.2f};"
          f" largest {m.max():.1f}; skewness {stats.skew(m):.1f}")